## Config

In [ ]:
import pandas as pd
import numpy as np
import re
import time
import os
from dotenv import load_dotenv
from tqdm.auto import tqdm


from huggingface_hub import login

import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM


In [ ]:
raw_path = os.path.join("data", "raw", 'projetos-lattes.csv')
dt = pd.read_csv(raw_path, sep=';', encoding='utf-8-sig')
dt.head()

In [ ]:
dt["descricao"][0]

In [ ]:
t = dt.loc[dt["projeto"].str.contains("Edital")]
t["projeto"].values[5]

In [ ]:
dt.info()

In [ ]:
dt.shape

In [ ]:
dt.isna().sum()

In [ ]:
df = dt.copy(deep=True)

## Filtering to only projects run in 2025 and 2026

In [ ]:
df["ano_conclusao"].unique()

In [ ]:
df["ano_conclusao"] = np.where(df["ano_conclusao"] == "Atual", np.nan, df["ano_conclusao"])
df["ano_conclusao"] = pd.to_numeric(df["ano_conclusao"])
df.head()

In [ ]:
current = df.loc[df["ano_conclusao"].isna()]
print(f"{current.shape}\n")
current.head()

### 2026

In [ ]:
df["ano_inicio"].unique()

In [ ]:
on_going_df = df.loc[df["ano_inicio"] == 2026]
print(f"{on_going_df.shape}\n")
on_going_df.head()

In [ ]:
finished_this_year = df.loc[df["ano_conclusao"] == 2026]
print(f"{finished_this_year.shape}\n")
finished_this_year.head()

In [ ]:
this_year_df = pd.concat([
    current,
    on_going_df,
    finished_this_year
])
print(f"{this_year_df.shape}\n")
this_year_df.head()

In [ ]:
this_year_df["ano_conclusao"].unique()

In [ ]:
this_year_df["ano_inicio"].unique()

### 2025

In [ ]:
# concluidos em 2025
fb = df.loc[df["ano_conclusao"] == 2025]
print(f"{fb.shape}\n")
fb.head()

In [ ]:
# em andamento em 2025
og = current.loc[current["ano_inicio"] < 2026]
print(f"{og.shape}\n")
og.head()

In [ ]:
last_year_df = pd.concat([
    finished_this_year,
    fb,
    og 
])
print(f"{last_year_df.shape}\n")
last_year_df.head(3)

In [ ]:
last_year_df["ano_inicio"].unique()

In [ ]:
last_year_df["ano_conclusao"].unique()

In [ ]:
# both
both_df = pd.concat([
    this_year_df,
    last_year_df
]).drop_duplicates().reset_index(drop=True)
print(f"{both_df.shape}\n")
both_df.to_csv(os.path.join("data", "output", "projetos_2025_2026.csv"), sep=";", encoding='utf-8-sig', index=False)

In [ ]:
to_process_df = pd.read_csv(os.path.join("data", "output", "to_process.csv"), sep=";", encoding='utf-8-sig')
to_process_df.shape

# Raw Data Extraction

### Extraction hard coded

lattes_id: ID Lattes do professor.

professor: Nome do docente.

titulo_projeto: Nome completo do projeto.

ano_inicio: Ano de início.

ano_conclusao: Ano de término (ou "Atual").

situacao: Em andamento / Concluído.

funcao: Coordenador / Integrante.

financiadores_lattes: Lista de agências/empresas financiadoras declaradas.

natureza: Pesquisa / Desenvolvimento / Extensão.

descricao_completa: Texto descritivo bruto do Lattes.

In [ ]:
def extrair_metadados_desc(texto):
    if not isinstance(texto, str):
        return pd.Series([None]*6)
    
    situacao = re.search(r'Situação:\s*([^;.]+)', texto)
    natureza = re.search(r'Natureza:\s*([^;.]+)', texto)
    alunos = re.search(r'Alunos envolvidos:\s*(.*?)(?=\.\s*Integrantes:|\.\s*Financiador|\Z)', texto)
    integrantes = re.search(r'Integrantes:\s*(.*?)(?=\.\s*Financiador|\Z)', texto)
    financiadores = re.search(r'Financiador\(es\):\s*(.+)', texto)
    desc_pura = re.search(r'Descrição:\s*(.*?)(?=\.\s*Situação:|\Z)', texto)
    
    return pd.Series([
        situacao.group(1).strip() if situacao else None,
        natureza.group(1).strip() if natureza else None,
        alunos.group(1).strip() if alunos else None,
        integrantes.group(1).strip() if integrantes else None,
        financiadores.group(1).strip() if financiadores else None,
        desc_pura.group(1).strip() if desc_pura else texto
    ])

In [ ]:
def extrair_metadados_projeto(text):
    if not isinstance(text, str):
        return pd.Series([None]*1)
    
    edital = re.search(r'\((Edital.*?)\)', text)

    return pd.Series([
        edital.group(1).split("Edital ")[1] if edital else None
    ])

In [91]:
to_process_df.isna().sum()

In [92]:
to_process_df[['situacao', 'natureza', 'alunos_envolvidos', 'integrantes', 'financiadores', 'descricao_limpa']] = to_process_df['descricao'].apply(extrair_metadados_desc)
to_process_df.head()

In [93]:
to_process_df[["edital"]] = to_process_df['projeto'].apply(extrair_metadados_projeto)
to_process_df.head()

In [94]:
to_process_df["edital"].unique()

In [95]:
to_process_df["ano_conclusao"].unique()

#### Contagem alunos

In [96]:
def extrair_contagem_alunos(texto):
    if not isinstance(texto, str):
        return pd.Series([0, 0, 0, 0])
    
    grad = re.search(r'Graduação:\s*\(([0-9]+)\)', texto)
    mest_ac = re.search(r'Mestrado acadêmico:\s*\(([0-9]+)\)', texto)
    mest_prof = re.search(r'Mestrado profissional:\s*\(([0-9]+)\)', texto)
    dout = re.search(r'Doutorado:\s*\(([0-9]+)\)', texto)
    
    n_grad = int(grad.group(1)) if grad else 0
    n_mest = (int(mest_ac.group(1)) if mest_ac else 0) + (int(mest_prof.group(1)) if mest_prof else 0)
    n_dout = int(dout.group(1)) if dout else 0
    n_total = n_grad + n_mest + n_dout
    
    return pd.Series([n_grad, n_mest, n_dout, n_total])

In [97]:
to_process_df[['qtd_graduacao', 'qtd_mestrado', 'qtd_doutorado', 'total_alunos']] = to_process_df['descricao'].apply(extrair_contagem_alunos)

In [98]:
to_process_df.head()

,lattes_id,professor,ano_inicio,ano_conclusao,projeto,descricao,situacao,natureza,alunos_envolvidos,integrantes,financiadores,descricao_limpa,edital,qtd_graduacao,qtd_mestrado,qtd_doutorado,total_alunos
0,92226104911153,Bruno Pereira dos Santos,2024,NaN,Sistemas Resilientes para Ambientes Inteligent...,Situação: Em andamento; Natureza: Pesquisa. Al...,Em andamento,Pesquisa,Graduação: (4) / Mestrado acadêmico: (2) / Dou...,Bruno Pereira dos Santos - Integrante / Paulo ...,Conselho Nacional de Desenvolvimento Científic...,Situação: Em andamento; Natureza: Pesquisa. Al...,NaN,4,2,2,8
1,92226104911153,Bruno Pereira dos Santos,2024,NaN,Projeto ARGUS,Descrição: O aumento exponencial no volume de ...,Em andamento,Pesquisa,NaN,Bruno Pereira dos Santos - Integrante / Maycon...,NaN,O aumento exponencial no volume de dados produ...,NaN,0,0,0,0
2,92226104911153,Bruno Pereira dos Santos,2023,NaN,Aprendizado de Máquina Federado para Internet ...,Descrição: Os ecossistemas de Internet das Coi...,Em andamento,Pesquisa,Mestrado acadêmico: (2) Doutorado: (2),Bruno Pereira dos Santos - Integrante / Heitor...,Fundação de Amparo à Pesquisa do Estado de Min...,Os ecossistemas de Internet das Coisas (IoT) o...,NaN,0,2,2,4
3,92226104911153,Bruno Pereira dos Santos,2023,NaN,Big Data Geocode: Uma aplicação de inteligênci...,Situação: Em andamento; Natureza: Pesquisa. In...,Em andamento,Pesquisa,NaN,Bruno Pereira dos Santos - Coordenador / Robes...,NaN,Situação: Em andamento; Natureza: Pesquisa. In...,NaN,0,0,0,0
4,1827829018668226,Christina von Flach Garcia Chavez,2023,NaN,ChriS - Characterizing Sustainable Research So...,"Descrição: Coletar e analisar dados, construir...",Em andamento,Pesquisa,Graduação: (1) / Mestrado acadêmico: (1) / Dou...,Christina von Flach Garcia Chavez - Coordenado...,NaN,"Coletar e analisar dados, construir e testar m...",NaN,1,1,2,4


In [99]:
both_df[-30:]

,lattes_id,professor,ano_inicio,ano_conclusao,projeto,descricao
72,851148137941240,Tatiane Nogueira Rios,2022,NaN,"SI2 - Instituto de Ciência, Tecnologia e Inova...",Descrição: Este projeto foi aprovado no Edital...
73,851148137941240,Tatiane Nogueira Rios,2021,NaN,CeMEAI - Centro de Ciências Matemática Aplicad...,Descrição: O foco desta proposta é a transferê...
74,851148137941240,Tatiane Nogueira Rios,2021,NaN,Uma Plataforma Digital Aberta Multiusuário De ...,Descrição: Este projeto foi selecionado pela C...
75,5075736089100544,Cássio Vinicius Serafim Prazeres,2024,2026.0,Argus,"Descrição: No cenário atual, a segurança patri..."
76,5075736089100544,Cássio Vinicius Serafim Prazeres,2024,2026.0,Aprendizado para IoT e Dispositivos Federados ...,Descrição: Desenvolver um arcabouço para a exe...
77,5075736089100544,Cássio Vinicius Serafim Prazeres,2023,2026.0,Uma Plataforma Inteligente para Redução de Que...,Descrição: A queda de pacientes em ambientes h...
78,5075736089100544,Cássio Vinicius Serafim Prazeres,2016,2026.0,SOFT-IoT: Self-Organizing Fog of Things,Situação: Concluído; Natureza: Pesquisa. Aluno...
79,6271096128174325,Frederico Araujo Durão,2023,2026.0,"SI2 - Instituto de Ciência, Tecnologia e Inova...","Descrição: Instituto de Ciência, Inovação e Te..."
80,6271096128174325,Frederico Araujo Durão,2022,2026.0,Centro de excelência em capacitação Positivo T...,Descrição: O projeto tem como objetivo estabel...
81,6271096128174325,Frederico Araujo Durão,2020,2026.0,"TRANSFORMAÇÃO DIGITAL: indústria 4.0, conectiv...","Descrição: Transformação Digital, do Acordo de..."


### LLM feature extraction

area_pgcomp: Subárea principal de pesquisa no PGCOMP.

edital_ou_chamada: Nome do edital/chamada identificado no texto.

tem_fomento: Booleano (Sim/Não) ou agência identificada no texto.

palavras_chave_llm: Lista de palavras-chave padronizadas.

parceria_internacional: Booleano (Sim/Não) e país/instituição parceira.

resumo_executivo: Síntese concisa do objetivo.

bolsa: tipo da bolsa (Produtividade, Tecnologica, Outros)

In [100]:
# huggingface token
load_dotenv()
# print("token: ", os.getenv("HF_TOKEN"))

True

In [101]:
# confirm if the token belogs to the user
# from huggingface_hub import HfApi

# user_info = HfApi().whoami(token=os.getenv("HF_TOKEN"))
# print(f"Token ativo para a conta: {user_info['name']}")

In [102]:
model_id = "google/gemma-2-2b-it"
# model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float32,
    device_map="cpu",
    local_files_only=True
)
model

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Gemma2TextScaledWordEmbedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_lay

In [103]:
system = """
<role>
Você é um assistente acadêmico especialista em mineração de dados da Plataforma Lattes e análise de projetos de pesquisa do programa de mestrado de Ciência da Computação da UFBA.
</role>

<instrucoes>
1. Analise o texto do projeto fornecido e extraia ou deduza os campos definidos no schema JSON.
2. Idioma: Retorne todas as respostas em Português do Brasil.
3. Ausência de Informação: Se uma informação não estiver presente e não puder ser inferida com segurança, atribua o valor null ao campo (nunca invente dados).
4. Formato de Saída: Responda APENAS com o objeto JSON válido, sem texto explicativo antes ou depois, e sem marcações de código markdown (como ```json).
</instrucoes>

<taxonomia_pgcomp>
Para o campo 'area_pgcomp', escolha APENAS UMA das seguintes áreas do PGCOMP/UFBA que melhor descreve o projeto:
- Engenharia de Software
- Inteligência Artificial e Aprendizado de Máquina
- Redes de Computadores e Sistemas Distribuídos
- Ciência de Dados e Banco de Dados
- Segurança da Informação e Criptografia
- Visualização de Dados e Processamento de Imagens
- Computação Aplicada e Sistemas Interativos
- Otimização e Teoria da Computação
- Outra
</taxonomia_pgcomp>


<schema_json>
{
  "edital_ou_chamada": "string | null (Nome do edital ou chamada pública citada no texto, ex: 'Edital Universal CNPq 09/2023')",
  "coordenador": "string | null (Nome do coordenador do projeto. Extraia da lista de integrantes se houver a marcação 'Coordenador')",
  "valor_financiamento": "string | null (valor identificado no próprio texto caso apareça)",
  "financiador": "array | null (Lista com os nomes das agências de fomento, bolsas ou instituições financiadoras, ex: ['CNPq', 'CAPES'])",
  "tipo_bolsa": "Produtividade | Formação Acadêmica | Desenvolvimento Tecnológico | Apoio Técnico | Fixação Regional | null",
  "area_pgcomp": "string (Apenas uma área contida na <taxonomia_pgcomp>)",
  "topico_pesquisa": "string (Tema/tópico específico do projeto, ex: 'Testes e Qualidade de Software')",
  "palavras_chave": ["array com 3 a 5 palavras-chave técnicas em Português"],
  "cooperacao_internacional": "boolean (true se houver colaboração com universidades/instituições estrangeiras, senão false)",
  "resumo_executivo": "string (Resumo conciso de 1 a 2 frases com o objetivo principal do projeto)"
}
</schema_json>

<exemplo>
Entrada do Usuário:
"Descrição: O projeto visa investigar test smells no contexto dos testes manuais e automatizados de software para propor técnicas e estratégias. (Chamada CNPq 14/2023 Apoio a Projetos Internacionais). Integrantes: Fulano de Tal - Coordenador / Beltrano - Integrante. Financiador(es): CNPq - Auxílio financeiro."

Saída Esperada:
{
  "edital_ou_chamada": "CNPq 14/2023 Apoio a Projetos Internacionais",
  "coordenador": "Fulano de Tal",
  "financiador": ["CNPq"],
  "valor_financiamento": "R$ 2.400,00",
  "tipo_bolsa": "Produtividade",
  "area_pgcomp": "Engenharia de Software",
  "topico_pesquisa": "Teste de Software",
  "palavras_chave": ["Test Smells", "Testes de Software", "Qualidade de Software"],
  "cooperacao_internacional": true,
  "resumo_executivo": "O projeto investiga a ocorrência de test smells em testes manuais e automatizados de software para propor métodos e estratégias para diminuição do seu impacto na qualidade."
}
</exemplo>
"""

In [104]:
def processar_texto_llm(texto: str) -> dict:
    prompts = []
    messages = [
        {"role": "user", "content": f"{system}\n\nProjeto:\n{texto}"}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompts.append(prompt)
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cpu")
    
    output = model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id
    )
    # print("output: ", output)
    # print(type(output))
    input_len = inputs.input_ids.shape[1]

    resposta_bruta = tokenizer.decode(
        output[0][input_len:], skip_special_tokens=True
    ).strip()

    # print(f"resposta bruta:\n{resposta_bruta}\n")
    try:
        match = re.search(r"\{.*\}", resposta_bruta, re.DOTALL)
        if match:
            dados = json.loads(match.group(0))
        else:
            dados = {}
    except Exception as e:
        print(f"Erro ao decodificar JSON: {e}")
        dados = {}

    return dados

In [105]:
data_to_extract = to_process_df.copy(deep=True)
data_to_extract.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   lattes_id          50 non-null     int64  
 1   professor          50 non-null     str    
 2   ano_inicio         50 non-null     int64  
 3   ano_conclusao      24 non-null     float64
 4   projeto            50 non-null     str    
 5   descricao          50 non-null     str    
 6   situacao           50 non-null     str    
 7   natureza           50 non-null     str    
 8   alunos_envolvidos  32 non-null     str    
 9   integrantes        50 non-null     str    
 10  financiadores      20 non-null     str    
 11  descricao_limpa    50 non-null     str    
 12  edital             8 non-null      str    
 13  qtd_graduacao      50 non-null     int64  
 14  qtd_mestrado       50 non-null     int64  
 15  qtd_doutorado      50 non-null     int64  
 16  total_alunos       50 non-null     int6

In [106]:
# data_to_extract = data_to_extract[2
textos_limpos = (
    data_to_extract["projeto"].astype(str)
    + " "
    + data_to_extract["descricao"].astype(str)
).tolist()

resultados_llm = []
arquivo_checkpoint = "checkpoint_resultados.jsonl"

print(f"Processando {len(textos_limpos)} projetos...")

loop_progress = tqdm(range(len(textos_limpos)), desc="Processando Projetos")

for i in loop_progress:
    texto = textos_limpos[i]
    
    init = time.time()
    res_lote = processar_texto_llm(texto)
    resultados_llm.append(res_lote)
    fim = time.time()
    
    tempo = fim - init
    
    loop_progress.set_postfix(tempo_lote=f"{tempo:.1f}s")
    
    # Gravação incremental
    with open(arquivo_checkpoint, "a", encoding="utf-8") as f:
        for res in res_lote:
            f.write(json.dumps(res, ensure_ascii=False) + "\n")

# --- ETAPA 4: MERGE FINAL ---
data_to_save = data_to_extract.copy(deep=True)
df_llm = pd.DataFrame(resultados_llm)
df_final = pd.concat([data_to_save, df_llm], axis=1)

df_final.head()

Processando 50 projetos...


Processando Projetos:   0%|          | 0/50 [00:00<?, ?it/s]

,lattes_id,professor,ano_inicio,ano_conclusao,projeto,descricao,situacao,natureza,alunos_envolvidos,integrantes,...,edital_ou_chamada,coordenador,valor_financiamento,financiador,tipo_bolsa,area_pgcomp,topico_pesquisa,palavras_chave,cooperacao_internacional,resumo_executivo
0,92226104911153,Bruno Pereira dos Santos,2024,NaN,Sistemas Resilientes para Ambientes Inteligent...,Situação: Em andamento; Natureza: Pesquisa. Al...,Em andamento,Pesquisa,Graduação: (4) / Mestrado acadêmico: (2) / Dou...,Bruno Pereira dos Santos - Integrante / Paulo ...,...,NaN,Paulo H. L. Rettore,NaN,[Conselho Nacional de Desenvolvimento Científi...,NaN,Ciência de Dados e Banco de Dados,Sistemas Resilientes,"[Sistemas Resilientes, Qualidade de Dados, Con...",None,O projeto aborda a qualidade dos dados e a con...
1,92226104911153,Bruno Pereira dos Santos,2024,NaN,Projeto ARGUS,Descrição: O aumento exponencial no volume de ...,Em andamento,Pesquisa,NaN,Bruno Pereira dos Santos - Integrante / Maycon...,...,NaN,Gustavo Bittencourt Figueiredo,NaN,[],Produtividade,Redes de Computadores e Sistemas Distribuídos,Computação Quântica,"[Computação Quântica, Veículos Autônomos, Dado...",None,O projeto visa desenvolver um framework para a...
2,92226104911153,Bruno Pereira dos Santos,2023,NaN,Aprendizado de Máquina Federado para Internet ...,Descrição: Os ecossistemas de Internet das Coi...,Em andamento,Pesquisa,Mestrado acadêmico: (2) Doutorado: (2),Bruno Pereira dos Santos - Integrante / Heitor...,...,NaN,Heitor S Ramos,NaN,[Fundação de Amparo à Pesquisa do Estado de Mi...,Formação Acadêmica,Inteligência Artificial e Aprendizado de Máquina,Aprendizado Federado para Internet das Coisas,"[Aprendizado Federado, Internet das Coisas, Mo...",None,O projeto investiga a extração de conhecimento...
3,92226104911153,Bruno Pereira dos Santos,2023,NaN,Big Data Geocode: Uma aplicação de inteligênci...,Situação: Em andamento; Natureza: Pesquisa. In...,Em andamento,Pesquisa,NaN,Bruno Pereira dos Santos - Coordenador / Robes...,...,NaN,Bruno Pereira dos Santos,NaN,[],NaN,Inteligência Artificial e Aprendizado de Máquina,Geocodificação de Eventos em Saúde,"[Geocodificação, Eventos em Saúde, Processamen...",None,A aplicação de inteligência computacional e pr...
4,1827829018668226,Christina von Flach Garcia Chavez,2023,NaN,ChriS - Characterizing Sustainable Research So...,"Descrição: Coletar e analisar dados, construir...",Em andamento,Pesquisa,Graduação: (1) / Mestrado acadêmico: (1) / Dou...,Christina von Flach Garcia Chavez - Coordenado...,...,NaN,Christina von Flach Garcia Chavez,NaN,[],Produtividade,Engenharia de Software,Software de Pesquisa,"[Software de Pesquisa, Software Sustentável, D...",None,O projeto investiga desafios relacionados ao d...


In [107]:
print(df_final.shape)
print(df_final.columns.tolist())
save_path = os.path.join("data", "output")
if not os.path.exists(save_path):
    os.mkdir(save_path)
df_final.to_csv(os.path.join(save_path, 'projetos_lattes_3.csv'), sep=';', index=False, encoding='utf-8-sig')
print("Processamento concluído com sucesso!")

(50, 27)
['lattes_id', 'professor', 'ano_inicio', 'ano_conclusao', 'projeto', 'descricao', 'situacao', 'natureza', 'alunos_envolvidos', 'integrantes', 'financiadores', 'descricao_limpa', 'edital', 'qtd_graduacao', 'qtd_mestrado', 'qtd_doutorado', 'total_alunos', 'edital_ou_chamada', 'coordenador', 'valor_financiamento', 'financiador', 'tipo_bolsa', 'area_pgcomp', 'topico_pesquisa', 'palavras_chave', 'cooperacao_internacional', 'resumo_executivo']
Processamento concluído com sucesso!
